In [1]:
# Load Huggingface Dataset
from datasets import load_from_disk

data_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_90m/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10.dataset'
dataset = load_from_disk(data_path)
dataset

/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['rel_x_coord', 'rel_y_coord', 'cell_id', 'batch_token', 'gene_panel_token', 'assay_token', 'species_token', 'tissue_token', 'batch_value', 'gene_panel_value', 'assay_value', 'species_value', 'tissue_value', 'gene_tokens', 'gene_expr', 'seg_tokens', 'n_nonzero_tokens'],
    num_rows: 88626282
})

In [7]:
def fix_seg_tokens(batch):
    seg = batch["seg_tokens"]
    arr = np.asarray(seg)

    # If sequences are uniform length, vectorize across the whole batch
    if arr.dtype != object:
        fixed = np.where(arr != 0, arr - 104, arr)
        return {"seg_tokens": fixed.tolist()}

    # Fallback for ragged sequences
    out = []
    for row in seg:
        a = np.asarray(row)
        a = np.where(a != 0, a - 104, a)
        out.append(a.tolist())
    return {"seg_tokens": out}

# Tune num_proc to your CPU count; tune batch sizes to your I/O
ds_fixed = ds.map(
    fix_seg_tokens,
    batched=True,
    batch_size=1000,
    writer_batch_size=10_000,
    num_proc=8,                 # e.g., 8–32 depending on your node
    load_from_cache_file=False, # force recompute
    desc="Adjusting seg_tokens (non-zero -> -104)"
)

new_path = f"{data_path}_new"
ds_fixed.save_to_disk(new_path, num_shards=4000)
print("Saved to:", new_path)

Adjusting seg_tokens (non-zero -> -104) (num_proc=8): 100%|████| 88626282/88626282 [14:25:01<00:00, 1707.60 examples/s]
Saving the dataset (0/32 shards):   0%|                           | 248000/88626282 [01:03<6:17:10, 3905.22 examples/s]

KeyboardInterrupt



In [ ]:
new_path = data_path.replace('.dataset', '_new.dataset')
ds_fixed.save_to_disk(new_path, num_shards=4000)
print("Saved to:", new_path)

In [24]:
from datasets import load_from_disk

ds = load_from_disk(new_path)

In [25]:
ds

Dataset({
    features: ['rel_x_coord', 'rel_y_coord', 'cell_id', 'batch_token', 'gene_panel_token', 'assay_token', 'species_token', 'tissue_token', 'batch_value', 'gene_panel_value', 'assay_value', 'species_value', 'tissue_value', 'gene_tokens', 'gene_expr', 'seg_tokens', 'n_nonzero_tokens'],
    num_rows: 88626282
})

In [ ]:
import numpy as np

def _reorder_pair(xs, ys):
    # Stable sort by squared radius without NumPy overhead
    order = sorted(range(len(xs)), key=lambda i: xs[i]*xs[i] + ys[i]*ys[i])
    return [xs[i] for i in order], [ys[i] for i in order]

def reorder_batch(batch):
    rx, ry = batch["rel_x_coord"], batch["rel_y_coord"]
    new_x, new_y = [], []
    append_x, append_y = new_x.append, new_y.append  # tiny micro-opt
    for xs, ys in zip(rx, ry):
        xs_sorted, ys_sorted = _reorder_pair(xs, ys)
        append_x(xs_sorted); append_y(ys_sorted)
    batch["rel_x_coord"] = new_x
    batch["rel_y_coord"] = new_y
    return batch

# Tune these for your machine and IO:
NUM_PROC = 8          # e.g., number of CPU cores
READ_BATCH = 20000     # how many rows processed per function call
WRITE_BATCH = 80000    # how many rows written per Arrow chunk

ds = ds.map(
    reorder_batch,
    batched=True,
    batch_size=READ_BATCH,
    writer_batch_size=WRITE_BATCH,
    num_proc=NUM_PROC,
    desc="Reordering rel_x_coord / rel_y_coord by radius"
)

Reordering rel_x_coord / rel_y_coord by radius (num_proc=8):   0%| | 0/88626282 

In [ ]:
item = ds[0]

In [ ]:
item['rel_x_coord']

In [ ]:
item['rel_y_coord']

In [ ]:
ds.save_to_disk(data_path, num_shards=4000)
print("Saved to:", data_path)

In [ ]:
ds_fixed

In [ ]:
item = ds_fixed[0]

In [11]:
import pickle

with open('/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_loader_idx.pkl', "rb") as f: 
    indices = pickle.load(f)

NameError: name 'pickle' is not defined

In [ ]:
ds_ordered = ds_fixed.select(indices)

In [ ]:
new_path = data_path.replace('.dataset', '_ordered.dataset')
ds_ordered.save_to_disk(new_path, num_shards=4000)
print("Saved to:", new_path)